# Iterative LOCO: Numerical Baseline and Analytical Integration Plan

**Numerical baseline and analytical-integration assessment**

The previous notebooks compared Jacobian matrices. This notebook moves to an iterative correction workflow.

Two goals must remain separate:

1. establish the current pyLOCO numerical baseline;
2. design and validate a true analytical-Jacobian integration.

A static analytical matrix is not the same as recalculating the analytical Jacobian at every iteration.

## Important current API limitation

pyLOCO currently recalculates its numerical quadrupole Jacobian internally. It can also load a Jacobian file, but that file is static. There is no public callback such as

```python
jacobian_provider(current_ring, configuration)
```

that pyLOCO calls at each iteration.

Therefore this notebook does **not** claim that loading one nominal analytical matrix is a full iterative analytical LOCO implementation. Adding a callback or a documented Jacobian-provider interface is a later package milestone.

In [1]:
%matplotlib inline

import copy
from tempfile import TemporaryDirectory

import at
import matplotlib.pyplot as plt
import numpy as np

from pyLOCO.analysis import plot_matrices
from pyLOCO.config import FitInitConfig, RMConfig
from pyLOCO.pyloco import pyloco
from pyLOCO.response_matrix import response_matrix

## 1. Nominal model and distorted machine

The machine contains two assigned quadrupole errors. The ORM from this copied lattice is treated as the measured ORM.

In [2]:
# Reference calculation
QF = at.Quadrupole("QF", 0.5, 1.2)
QD = at.Quadrupole("QD", 0.5, -1.2)
Dr, HalfDr = at.Drift("Dr", 0.5), at.Drift("Dr", 0.25)
Bend = at.Dipole("Bend", 1.0, 2*np.pi/40)
cell = at.Lattice([
    HalfDr, Bend, Dr, at.Monitor("BPM_F"),
    at.Corrector("HCOR_F", 0, [0, 0]), at.Corrector("VCOR_F", 0, [0, 0]), QF,
    Dr, Bend, Dr, at.Monitor("BPM_D"),
    at.Corrector("HCOR_D", 0, [0, 0]), at.Corrector("VCOR_D", 0, [0, 0]), QD, HalfDr,
], name="Simple FODO cell with diagnostics", energy=1e9)
model = cell * 20
machine = copy.deepcopy(model)

bpm_indices = np.asarray(at.get_refpts(model, at.Monitor), int)
hcor_indices = np.asarray(at.get_refpts(model, "HCOR*"), int)
vcor_indices = np.asarray(at.get_refpts(model, "VCOR*"), int)
quad_indices = np.asarray(at.get_refpts(model, at.Quadrupole), int)
n_bpm, n_hcor, n_vcor = len(bpm_indices), len(hcor_indices), len(vcor_indices)

assigned_positions = np.asarray([0, 11])
assigned_indices = quad_indices[assigned_positions]
assigned_delta_K = np.asarray([+0.012, -0.009])
for qidx, delta in zip(assigned_indices, assigned_delta_K):
    new_K = float(machine[int(qidx)].PolynomB[1]) + float(delta)
    machine[int(qidx)].PolynomB[1] = new_K
    machine[int(qidx)].K = new_K

corrector_kick_rad = 1e-5
kick_steps = [[corrector_kick_rad]*n_hcor, [corrector_kick_rad]*n_vcor]
rm_config = RMConfig(
    bpm_ords=bpm_indices, cm_ords=[hcor_indices, vcor_indices],
    dkick=kick_steps, calculator="Linear", includeDispersion=False,
    HCMCoupling=np.zeros(n_hcor), VCMCoupling=np.zeros(n_vcor),
)
R_model = response_matrix(model, config=rm_config)
R_measured = response_matrix(machine, config=rm_config)
print("ORM shape:", R_model.shape)
print("Assigned errors [m^-2]:", assigned_delta_K)
print("Initial normalized ORM RMS [m/rad]:",
      np.sqrt(np.mean(((R_measured-R_model)/corrector_kick_rad)**2)))

ORM shape: (80, 80)
Assigned errors [m^-2]: [ 0.012 -0.009]
Initial normalized ORM RMS [m/rad]: 0.15935121101272323


## 2. Current pyLOCO numerical baseline

Fit only the two quadrupoles that contain assigned errors. This keeps the milestone transparent.

pyLOCO uses its internal forward-difference Jacobian and updates it during the iterative fit.

In [3]:
# Reference calculation
fit_config = FitInitConfig(
    fit_list=["quads"], CMstep=kick_steps, individuals=True,
    quads_attr="PolynomB", quads_attr_index=1,
)
equal_weights = np.ones(2*n_bpm)

In [4]:
# Reference calculation
with TemporaryDirectory(prefix="fodo_iterative_numerical_") as output:
    numerical_result = pyloco(
        copy.deepcopy(model),
        algorithm="lm", nIter=3,
        used_bpms_ords=bpm_indices,
        used_cor_ords=[hcor_indices, vcor_indices],
        quads_ords=assigned_indices,
        skew_ords=np.array([], dtype=int),
        CAVords=np.array([], dtype=int),
        nHBPM=n_bpm, nVBPM=n_bpm,
        nHorCOR=n_hcor, nVerCOR=n_vcor,
        quads_tilt_ind=assigned_indices,
        orm_measured=R_measured,
        weights=equal_weights,
        includeDispersion=False,
        measured_eta_x=np.zeros(n_bpm),
        measured_eta_y=np.zeros(n_bpm),
        CMstep=kick_steps, rfStep=1.0, Frequency=1.0,
        fit_list=["quads"], quad_individuals=True,
        remove_coupling_=False,
        outlier_rejection=False, apply_normalization=False,
        svd_selection_method="threshold", svd_threshold=1e-12,
        show_svd_plot=False, nLMIter=4,
        Starting_Lambda=1e-3, max_lm_lambda=15,
        scaled=True, plot_fit_parameters=False,
        auto_correct_delta=False, fixedpathlength=False,
        fixedmomentum=False, output_dir=output,
        fit_cfg=fit_config,
    )

fit_parameters, fit_dictionary, fitted_model, R_fitted, _, chi2_history, _, _ = numerical_result
fitted_delta_K = np.asarray([
    fitted_model[int(qidx)].PolynomB[1] - model[int(qidx)].PolynomB[1]
    for qidx in assigned_indices
], dtype=float)
print("Assigned dK:", assigned_delta_K)
print("Fitted dK:  ", fitted_delta_K)
print("Fit error:  ", fitted_delta_K-assigned_delta_K)


==== Iteration 1/3 – LM ====
[Jacobian] Computing normal-quadrupole Jacobian (iteration 1)...


Normal quad Jacobian: 1.2 s
[Jacobian] Saved normal-quadrupole Jacobian to /var/folders/vj/crrgrwws3s902yfns0l06_rw0000gp/T/fodo_iterative_numerical_0hwcmxzo/jacobians/quads/J_quads_iter1_1e-05urad_-3000.0Hz.h5
Initial Chi²: 2.5401e-12
  LM inner 1: chi² 6.2609e-15 (previous 2.5401e-12), λ=0.001
Chi² after correction: 6.2609e-15

==== Iteration 2/3 – LM ====
[Jacobian] Computing normal-quadrupole Jacobian (iteration 2)...


Normal quad Jacobian: 1.3 s
Initial Chi²: 6.2609e-15
  LM inner 1: chi² 1.2486e-18 (previous 6.2609e-15), λ=0.001
Chi² after correction: 1.2486e-18

==== Iteration 3/3 – LM ====
[Jacobian] Computing normal-quadrupole Jacobian (iteration 3)...


Normal quad Jacobian: 1.1 s
Initial Chi²: 1.2486e-18
  LM inner 1: chi² 1.5141e-22 (previous 1.2486e-18), λ=0.001
Chi² after correction: 1.5141e-22
LOCO LM completed! :).
Assigned dK: [ 0.012 -0.009]
Fitted dK:   [ 0.01200001 -0.00900006]
Fit error:   [ 1.32824996e-08 -6.14360057e-08]


## 3. Apply the numerical correction

The fitted values describe machine errors. The correction has the opposite sign:

$$K_{\mathrm{corrected}}=K_{\mathrm{machine}}-\Delta K_{\mathrm{fit}}$$

In [5]:
# ==========================================================
# Open analysis — implementation required
# ==========================================================
#
# 1. Copy machine.
# 2. Subtract fitted_delta_K at assigned_indices.
# 3. Recalculate the ORM.
# 4. Report ORM RMS before and after correction.
# 5. Compare tunes and beta beating.
#
# Analysis implementation

## 4. What a true analytical iterative fit requires

At iteration \(i\), the Jacobian must be evaluated at the current model:

$$J_i=J\left(K_i\right)$$

and the update solves approximately

$$J_i\,\Delta K_i \simeq R_{\mathrm{measured}}-R(K_i)$$

Then the model changes:

$$K_{i+1}=K_i+\Delta K_i$$

A static file containing \(J(K_0)\) does not satisfy this requirement after the first update.

## 5. Analytical Jacobian provider interface

Do not modify pyLOCO yet. First write the scientific interface that would be needed.

Suggested inputs:

- current lattice,
- BPM indices,
- H/V corrector indices,
- fitted quadrupole indices,
- ORM kick and normalization convention.

Expected output:

- shape `(number of ORM measurements, number of fitted parameters)`,
- derivative with respect to \(K\), not \(KL\),
- the same Fortran flattening and block ordering as pyLOCO.

In [6]:
# ==========================================================
# Open interface analysis — implementation required
# ==========================================================
#
# Sketch:
#
# def analytical_jacobian_provider(
#     current_ring,
#     bpm_indices,
#     hcor_indices,
#     vcor_indices,
#     fitted_quad_indices,
#     corrector_kick_rad,
# ):
#     ...
#     return J_analytical
#
# Do not integrate it into pyLOCO until its output is validated.

## 6. Validation gate before integration

For several model states, compare the proposed provider with pyLOCO's numerical Jacobian.

Required checks:

1. raw maximum difference,
2. RMS difference,
3. relative norm difference,
4. separate \(R_{xx}\) and \(R_{yy}\) checks,
5. K/KL conversion,
6. signs,
7. kick normalization,
8. parameter and measurement order,
9. timing,
10. behaviour after one or more model updates.

Do not approve integration only because plots look similar.

In [7]:
# ==========================================================
# Open comparison analysis — implementation required
# ==========================================================
#
# Test at:
# 1. nominal model,
# 2. assigned-error model,
# 3. model after one numerical LOCO update,
# 4. model close to convergence.
#
# Record numerical metrics and timing for every state.

## 7. Possible pyLOCO integration design

A future package change could expose one of these options:

```python
jacobian_method="numerical"
jacobian_method="analytical"
```

with an internal provider called at every iteration, or a callable:

```python
jacobian_provider(current_ring, context)
```

The provider must return the documented pyLOCO convention. Backward compatibility should keep the numerical method available.

## 8. Final comparison

| Quantity | Numerical iterative LOCO | Analytical iterative LOCO |
|---|---:|---:|
| Iterations | ? | ? |
| Final ORM RMS | ? | ? |
| Quadrupole fit error | ? | ? |
| Beta-beating RMS | ? | ? |
| Jacobian time | ? | ? |
| Total fit time | ? | ? |

The analytical column remains incomplete until a genuine per-iteration provider exists.

## 9. Scientific and implementation checks

1. Why must the analytical Jacobian be recalculated after a model update?
2. When might a static Jacobian still converge?
3. Why is convergence alone not enough for validation?
4. Which pyLOCO convention should a provider guarantee?
5. Should numerical calculation remain as a fallback?
6. How should analytical failures be reported?
7. What tests are required before making analytical calculation the default?

## 10. Results and conclusions

### Numerical baseline
...

### Correction quality
...

### Missing analytical API
...

### Proposed provider
...

### Validation required before integration
...

### Recommendation
...